# Example queries: `annual` (resstock_oedi)

Auto-generated from `tests/query_snapshots/annual.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [1]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [2]:
# This notebook lives in `tests/example_notebooks/`; the snapshot test
# cache is its sibling `tests/query_snapshots/resstock_oedi_cache/`. Resolve
# the path relative to the notebook directory (`_dh[0]` is set by
# IPython at kernel startup; falls back to CWD outside Jupyter).
_NB_DIR = Path(_dh[0] if "_dh" in globals() else ".").resolve()
_CACHE = (_NB_DIR / "../query_snapshots/resstock_oedi_cache").resolve()
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


INFO:buildstock_query.query_core:Loading resstock_2024_amy2018_release_2 ...


INFO:botocore.tokens:Loading cached SSO token for nrel-sso


## `annual_totals_overall`

Annual electricity + natural gas totals restricted to CO.


In [3]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption', 'out.natural_gas.total.energy_consumption'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,sample_count,units_count,electricity.total.energy_consumption,natural_gas.total.energy_consumption
0,9425,2.377943e+06,2.259390e+10,3.795731e+10


## `annual_totals_by_building_type`

Annual totals grouped by building type, CO only. Two variants confirm that omitting annual_only (default True) matches passing annual_only=True explicitly.


In [4]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption', 'out.natural_gas.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,sample_count,units_count,electricity.total.energy_consumption,natural_gas.total.energy_consumption
0,Mobile Home,391,9.864994e+04,7.898215e+08,1.279629e+09
1,Multi-Family with 2 - 4 Units,469,1.183295e+05,8.547447e+08,1.054712e+09
2,Multi-Family with 5+ Units,2001,5.048556e+05,3.249347e+09,2.466138e+09
3,Single-Family Attached,664,1.675283e+05,1.345615e+09,1.909135e+09
4,Single-Family Detached,5900,1.488580e+06,1.635437e+10,3.124770e+10


## `annual_electricity_by_vintage_sort`

Annual electricity grouped by vintage with sort, limit 5, CO only.


In [5]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['vintage'],
    restrict=[('state', ['CO'])],
    sort=True,
    limit=5,
)
result.head() if hasattr(result, 'head') else result


,vintage,sample_count,units_count,electricity.total.energy_consumption
0,1940s,240,60552.393295,5.792687e+08
1,1950s,702,177115.750389,1.651332e+09
2,1960s,805,203102.819178,1.999671e+09
3,1970s,1755,442789.375972,4.193620e+09
4,1980s,1362,343634.831951,3.374342e+09


## `annual_electricity_by_vintage_nosort`

Annual electricity by vintage with sort=false + limit. Hits the no-ORDER-BY branch which existing entries never exercise. Marked nondeterministic because LIMIT without ORDER BY makes Trino free to return any 5 rows; only SQL shape is meaningful.


In [6]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['vintage'],
    restrict=[('state', ['CO'])],
    sort=False,
    limit=5,
)
result.head() if hasattr(result, 'head') else result


,vintage,sample_count,units_count,electricity.total.energy_consumption
0,1990s,1579,398384.287555,3.619632e+09
1,1960s,805,203102.819178,1.999671e+09
2,<1940,720,181657.179886,1.638828e+09
3,1950s,702,177115.750389,1.651332e+09
4,2010s,619,156174.714374,1.440811e+09


## `annual_nonzero_count_natural_gas`

Annual natural gas with nonzero_units_count requested, CO only.


In [7]:
result = bsq.query(
    enduses=['out.natural_gas.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    get_nonzero_count=True,
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,sample_count,units_count,natural_gas.total.energy_consumption,natural_gas.total.energy_consumption__nonzero_units_count
0,Mobile Home,391,9.864994e+04,1.279629e+09,7.266287e+04
1,Multi-Family with 2 - 4 Units,469,1.183295e+05,1.054712e+09,9.259470e+04
2,Multi-Family with 5+ Units,2001,5.048556e+05,2.466138e+09,3.476717e+05
3,Single-Family Attached,664,1.675283e+05,1.909135e+09,1.382613e+05
4,Single-Family Detached,5900,1.488580e+06,3.124770e+10,1.211300e+06


## `annual_baseline_quartiles`

Annual baseline (upgrade=0) electricity with get_quartiles=true. Hits the quartile branch on the upgrade-0 path where there is no upgrade-table join — the existing quartile coverage is upgrade=1 only.


In [8]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    get_quartiles=True,
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,sample_count,units_count,electricity.total.energy_consumption,electricity.total.energy_consumption__upgrade__quartiles,electricity.total.energy_consumption__upgrade__nonzero_quartiles
0,Mobile Home,391,9.864994e+04,7.898215e+08,"[504.66838283656665, 717.6401988200189, 2139.8...","[504.66838283656665, 717.6401988200189, 2139.8..."
1,Multi-Family with 2 - 4 Units,469,1.183295e+05,8.547447e+08,"[383.3369597852667, 554.2193740259352, 2459.90...","[383.3369597852667, 554.2193740259352, 2459.90..."
2,Multi-Family with 5+ Units,2001,5.048556e+05,3.249347e+09,"[386.85381262733335, 595.3103499843787, 2071.2...","[386.85381262733335, 595.3103499843787, 2071.2..."
3,Single-Family Attached,664,1.675283e+05,1.345615e+09,"[438.7273920478167, 1148.4084783902204, 3661.2...","[438.7273920478167, 1148.4084783902204, 3661.2..."
4,Single-Family Detached,5900,1.488580e+06,1.635437e+10,"[363.9942691539, 1454.59166026229, 4354.394233...","[363.9942691539, 1454.59166026229, 4354.394233..."


## `annual_agg_func_mean`

Annual electricity with agg_func='mean'. Pins the non-sum aggregation branch (no weight multiplication, AVG instead of SUM).


In [9]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    agg_func='mean',
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,sample_count,units_count,electricity.total.energy_consumption__mean
0,Mobile Home,391,9.864994e+04,8006.304380
1,Multi-Family with 2 - 4 Units,469,1.183295e+05,7223.430772
2,Multi-Family with 5+ Units,2001,5.048556e+05,6436.190746
3,Single-Family Attached,664,1.675283e+05,8032.167041
4,Single-Family Detached,5900,1.488580e+06,10986.559115


## `annual_agg_func_max`

Annual electricity with agg_func='max'. Pins the MAX() aggregation branch.


In [10]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    agg_func='max',
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,sample_count,units_count,electricity.total.energy_consumption__max
0,Mobile Home,391,9.864994e+04,61903.936797
1,Multi-Family with 2 - 4 Units,469,1.183295e+05,35722.432743
2,Multi-Family with 5+ Units,2001,5.048556e+05,29942.485097
3,Single-Family Attached,664,1.675283e+05,37661.390944
4,Single-Family Detached,5900,1.488580e+06,83445.832739


## `annual_agg_func_min`

Annual electricity with agg_func='min'. Pins the MIN() aggregation branch.


In [11]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    agg_func='min',
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,sample_count,units_count,electricity.total.energy_consumption__min
0,Mobile Home,391,9.864994e+04,504.668383
1,Multi-Family with 2 - 4 Units,469,1.183295e+05,383.336960
2,Multi-Family with 5+ Units,2001,5.048556e+05,386.853813
3,Single-Family Attached,664,1.675283e+05,438.727392
4,Single-Family Detached,5900,1.488580e+06,363.994269


## `annual_with_extra_weight_sqft`

Annual electricity with weights=[$SQFT_COL] — resstock 'in.sqft', comstock 'in.sqft..ft2'. Pins the SQL shape where the user supplies an explicit additional weight column on top of the default sample_weight; Athena gets `electricity * sample_weight * sqft`. Internally exercised by utility queries (eiaid_weights.weight); no other entry covers user-supplied weights directly.


In [12]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    weights=['in.sqft'],
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,sample_count,units_count,electricity.total.energy_consumption
0,Mobile Home,391,1.169897e+08,1.013762e+12
1,Multi-Family with 2 - 4 Units,469,1.198344e+08,9.229353e+11
2,Multi-Family with 5+ Units,2001,4.532614e+08,3.137632e+12
3,Single-Family Attached,664,2.748445e+08,2.468911e+12
4,Single-Family Detached,5900,3.269609e+09,4.025909e+13


## `annual_agg_func_arbitrary`

Annual electricity with agg_func='arbitrary'. Pins the ARBITRARY() (Athena's any-value) aggregation branch. Marked nondeterministic because arbitrary() picks any group value per row; only SQL shape is meaningful.


In [13]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
    agg_func='arbitrary',
)
result.head() if hasattr(result, 'head') else result


,geometry_building_type_recs,sample_count,units_count,electricity.total.energy_consumption__arbitrary
0,Mobile Home,391,9.864994e+04,6044.004680
1,Multi-Family with 2 - 4 Units,469,1.183295e+05,7924.348666
2,Multi-Family with 5+ Units,2001,5.048556e+05,6609.924917
3,Single-Family Attached,664,1.675283e+05,6068.622650
4,Single-Family Detached,5900,1.488580e+06,8346.371007
